# Dataset 3 — Credit Card Fraud Detection (Google Colab)

Self-contained Colab port of the Dataset 3 pipeline.

**Pipeline:**
1. Load Kaggle Credit Card Fraud dataset via `kagglehub`
2. Clean + light EDA (3 plots: class distribution, time distribution, feature/Class correlation)
3. Time-based expanding-window splits + SMOTE (training only)
4. **Algorithm 1 — Logistic Regression** (full pipeline + 4 plots)
5. **Algorithm 2 — Random Forest** (placeholder — to be implemented)
6. **Algorithm 3 — Feed-Forward Neural Network** (full pipeline + 4 plots, GPU-aware)

**Metrics:** Precision, Recall, F1, MCC, AUC-ROC.  
**Artefacts:** PNGs + CSVs are also written to `./results/` for download.

> Tip: switch the Colab runtime to GPU (`Runtime → Change runtime type → GPU`) before running the FFNN section.

## Phase 1 — Environment setup

In [ ]:
# Install dependencies (Colab already ships pandas/numpy/sklearn/matplotlib/seaborn/torch).
!pip install -q "kagglehub[pandas-datasets]" imbalanced-learn

### Kaggle authentication

`kagglehub` needs Kaggle API credentials. The easiest path in Colab is to upload your `kaggle.json` (Account → Create New API Token):

```python
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
```

Or call `kagglehub.login()` interactively.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
)
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Shared constants
RANDOM_STATE = 42
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {DEVICE}")


def save_fig(fig, name):
    """Save figure to ./results AND display inline in the notebook."""
    path = os.path.join(RESULTS_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved: {path}")
    plt.show()
    plt.close(fig)

## Phase 2 — Load data from Kaggle

In [ ]:
# Load latest version of the dataset directly into a DataFrame.
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "whenamancodes/fraud-detection",
    "creditcard.csv",
)

print("First 5 records:")
print(df.head())

In [ ]:
# Cleaning + sanity checks
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)

df["Class"] = df["Class"].astype(int)

print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
print(f"\nBasic statistics:\n{df.describe()}")

missing = df.isnull().sum()
total_missing = missing.sum()
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])

duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    print("  -> Dropping duplicates")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  -> New shape: {df.shape[0]:,} rows")

## Phase 3 — Exploratory Data Analysis (3 retained plots)

In [ ]:
# ── Plot 1: Class distribution (severe imbalance) ──
print("=" * 60)
print("2. CLASS DISTRIBUTION")
print("=" * 60)

class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index() * 100

print(f"  Legitimate (0): {class_counts[0]:>8,}  ({class_pct[0]:.4f}%)")
print(f"  Fraudulent (1): {class_counts[1]:>8,}  ({class_pct[1]:.4f}%)")
print(f"  Imbalance ratio: 1 : {class_counts[0] / class_counts[1]:.0f}")

colors = ["#2ecc71", "#e74c3c"]
labels = ["Legitimate (0)", "Fraudulent (1)"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    axes[0].text(i, cnt + cnt * 0.01, f"{cnt:,}\n({pct:.3f}%)", ha="center", fontsize=10)

axes[1].pie(class_counts.values, labels=labels, colors=colors,
            autopct="%1.3f%%", startangle=90, explode=(0, 0.1))
axes[1].set_title("Class Proportion", fontsize=14, fontweight="bold")

fig.suptitle("Severe Class Imbalance in Dataset 3", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "01_class_distribution.png")

In [ ]:
# ── Plot 2: Transaction volume distribution over ~48 hours ──
print("=" * 60)
print("3. TIME FEATURE ANALYSIS")
print("=" * 60)

df["Hour"] = df["Time"] / 3600
total_hours = df["Hour"].max()
print(f"  Time range: 0 - {df['Time'].max():,.0f} seconds  "
      f"({total_hours:.1f} hours ~ {total_hours / 24:.1f} days)")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].hist(df["Hour"], bins=48, color="#3498db", edgecolor="black", alpha=0.7)
axes[0].set_title("All Transactions Over Time", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Transactions")

fraud_hours = df[df["Class"] == 1]["Hour"]
axes[1].hist(fraud_hours, bins=48, color="#e74c3c", edgecolor="black", alpha=0.7)
axes[1].set_title("Fraudulent Transactions Over Time", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Number of Transactions")
axes[1].set_xlabel("Time (Hours)")

fig.suptitle("Transaction Volume Distribution Over ~48 Hours",
             fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "02_time_distribution.png")

In [ ]:
# ── Plot 3: Feature correlation with the Class label ──
print("=" * 60)
print("4. CORRELATION ANALYSIS")
print("=" * 60)

v_cols = [f"V{i}" for i in range(1, 29)]
feature_cols = v_cols + ["Amount"]
corr_with_class = (
    df[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values()
)

print("  Features most negatively correlated with Class (fraud):")
for feat, corr in corr_with_class.head(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

print("\n  Features most positively correlated with Class (fraud):")
for feat, corr in corr_with_class.tail(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values,
        color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with Class", fontsize=12)
ax.set_title("Feature Correlation with Fraud Label", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.8)
fig.tight_layout()
save_fig(fig, "08_feature_correlation_with_class.png")

## Phase 4 — Preprocessing utilities (splits + SMOTE)

In [ ]:
# ── Feature / target split ──
print("=" * 60)
print("5. PREPROCESSING PIPELINE")
print("=" * 60)

df.drop(columns=["Hour"], inplace=True, errors="ignore")

X = df.drop(columns=["Class"])
y = df["Class"]

print(f"  Feature matrix X: {X.shape}")
print(f"  Target vector y:  {y.shape}  "
      f"(fraud={y.sum():,}, legit={len(y) - y.sum():,})")

In [ ]:
def create_time_splits(X, test_window=8, min_train_hours=16):
    """
    Expanding-window time-based train/test splits.
    The dataset spans ~48 hours; train always precedes test (no temporal leakage).
    """
    hours = X["Time"] / 3600
    total_hours = hours.max()

    splits = []
    test_start = min_train_hours

    while test_start < total_hours:
        test_end = min(test_start + test_window, total_hours + 1)
        train_mask = hours < test_start
        test_mask = (hours >= test_start) & (hours < test_end)

        train_idx = X[train_mask].index.tolist()
        test_idx = X[test_mask].index.tolist()

        if len(train_idx) > 0 and len(test_idx) > 0:
            splits.append((train_idx, test_idx))

        test_start += test_window

    return splits


def scale_and_resample(X_features, y, train_idx, test_idx):
    """
    Scale features (fit on train only) and apply SMOTE to the training set.
    Returns: X_train_res, y_train_res, X_test_scaled, y_test, scaler.
    """
    feature_names = X_features.columns.tolist()

    X_train_raw = X_features.loc[train_idx]
    X_test_raw = X_features.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=feature_names, index=X_train_raw.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=feature_names, index=X_test_raw.index,
    )

    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

    return X_train_res, y_train_res, X_test_scaled, y_test, scaler


splits = create_time_splits(X)

print("\n" + "=" * 60)
print("6. TIME-BASED TRAIN / TEST SPLITTING")
print("=" * 60)
hours = X["Time"] / 3600
print(f"  Total time span: {hours.max():.1f} hours")
print(f"  Training starts at: 16h | Test window: 8h")
print(f"  Number of time-based splits: {len(splits)}\n")

print(f"  {'Split':<8} {'Train Size':>12} {'Train Fraud':>13} "
      f"{'Test Size':>11} {'Test Fraud':>12} {'Train Hours':>13} {'Test Hours':>12}")
print(f"  {'-' * 81}")
for i, (train_idx, test_idx) in enumerate(splits):
    train_fraud = y.loc[train_idx].sum()
    test_fraud = y.loc[test_idx].sum()
    train_hours_range = f"0-{hours.loc[train_idx].max():.0f}h"
    test_hours_range = f"{hours.loc[test_idx].min():.0f}-{hours.loc[test_idx].max():.0f}h"
    print(
        f"  {i + 1:<8} {len(train_idx):>12,} {train_fraud:>13,} "
        f"{len(test_idx):>11,} {test_fraud:>12,} "
        f"{train_hours_range:>13} {test_hours_range:>12}"
    )

## Phase 5 — Evaluation helpers (shared by all models)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    """Standard metric suite (Precision, Recall, F1, MCC, AUC-ROC)."""
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "mcc":       matthews_corrcoef(y_true, y_pred),
        "auc_roc":   roc_auc_score(y_true, y_prob),
    }


def print_classification_report(y_true, y_pred):
    print(classification_report(
        y_true, y_pred,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0,
    ))


def print_summary_table(results_df, model_name):
    print(f"\n  -- {model_name} Summary Across All Splits --")
    print(f"  {'Metric':<12} {'Mean':>10} {'Std':>10}")
    print(f"  {'-' * 32}")
    for metric in ["precision", "recall", "f1", "mcc", "auc_roc"]:
        mean_val = results_df[metric].mean()
        std_val = results_df[metric].std()
        print(f"  {metric:<12} {mean_val:>10.4f} {std_val:>10.4f}")

In [ ]:
def plot_confusion_matrices(splits, X_features, y, train_and_predict_fn, model_name, filename):
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        y_pred = train_and_predict_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]

        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[split_num - 1],
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
        axes[split_num - 1].set_title(f"Split {split_num}", fontsize=12, fontweight="bold")
        axes[split_num - 1].set_ylabel("Actual")
        axes[split_num - 1].set_xlabel("Predicted")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontsize=16, fontweight="bold", y=1.02)
    fig.tight_layout()
    save_fig(fig, filename)


def plot_roc_curves(splits, X_features, y, train_and_predict_proba_fn, model_name, filename):
    fig, ax = plt.subplots(figsize=(8, 6))

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_and_predict_proba_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Split {split_num} (AUC={auc_val:.4f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} - ROC Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, filename)


def plot_metrics_bars(results_df, splits, model_name, filename):
    fig, ax = plt.subplots(figsize=(10, 5))

    metrics_to_plot = ["precision", "recall", "f1", "mcc", "auc_roc"]
    x = np.arange(len(splits))
    width = 0.15
    colors_metrics = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6", "#f39c12"]

    for i, metric in enumerate(metrics_to_plot):
        vals = results_df[metric].values
        ax.bar(x + i * width, vals, width,
               label=metric.upper().replace("_", "-"), color=colors_metrics[i])

    ax.set_xlabel("Split", fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"{model_name} - Metrics by Split", fontsize=14, fontweight="bold")
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f"Split {i+1}" for i in range(len(splits))])
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1.05)
    fig.tight_layout()
    save_fig(fig, filename)


def plot_feature_coefficients(coefs, feature_names, model_name, split_label, filename, top_n=15):
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(np.abs(coefs))[::-1]

    ax.barh(
        range(top_n),
        coefs[sorted_idx[:top_n]],
        color=["#e74c3c" if c < 0 else "#2ecc71" for c in coefs[sorted_idx[:top_n]]],
        edgecolor="black", linewidth=0.5,
    )
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]])
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Coefficients ({split_label})",
                 fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)

## Phase 6 — Algorithm 1: Logistic Regression

Linear baseline using L2-regularised logistic regression. SMOTE applied to training folds only.

Objective: $Z_1 = \min_{w,c} -\frac{1}{n}\sum_i [y_i \log\sigma(w^\top x_i + c) + (1-y_i)\log(1-\sigma(w^\top x_i + c))] + \lambda\|w\|_2^2$

In [ ]:
def _build_lr():
    """Fresh L2-penalised LogisticRegression instance."""
    return LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )


print("=" * 60)
print("7. LOGISTIC REGRESSION - TRAINING & EVALUATION")
print("=" * 60)

X_features = X.drop(columns=["Time"])
feature_names = X_features.columns.tolist()

lr_results = []
lr_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"\n  -- Split {split_num} --")

    X_train_res, y_train_res, X_test_scaled, y_test, _ = scale_and_resample(
        X_features, y, train_idx, test_idx,
    )
    print(f"    Train: {len(X_train_res):,} samples (after SMOTE)  |  "
          f"Test: {len(X_test_scaled):,} samples")

    model = _build_lr()
    model.fit(X_train_res, y_train_res)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    lr_results.append(metrics)

    print(f"    Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  "
          f"F1: {metrics['f1']:.4f}  |  MCC: {metrics['mcc']:.4f}  |  "
          f"AUC-ROC: {metrics['auc_roc']:.4f}")
    print_classification_report(y_test, y_pred)

    lr_last_model = model

lr_df = pd.DataFrame(lr_results)
lr_df.to_csv(os.path.join(RESULTS_DIR, "lr_results.csv"), index=False)
print_summary_table(lr_df, "Logistic Regression")

In [ ]:
# ── LR visualisations ──
print("=" * 60)
print("8. LOGISTIC REGRESSION - VISUALISATIONS")
print("=" * 60)

def _lf_train_predict(X_features, y, train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_and_resample(X_features, y, train_idx, test_idx)
    m = _build_lr()
    m.fit(X_tr, y_tr)
    return m.predict(X_te)

def _lf_train_predict_proba(X_features, y, train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_and_resample(X_features, y, train_idx, test_idx)
    m = _build_lr()
    m.fit(X_tr, y_tr)
    return m.predict_proba(X_te)[:, 1]

plot_confusion_matrices(splits, X_features, y, _lf_train_predict,
                        "Logistic Regression", "12_lr_confusion_matrices.png")
plot_roc_curves(splits, X_features, y, _lf_train_predict_proba,
                "Logistic Regression", "13_lr_roc_curves.png")
plot_metrics_bars(lr_df, splits, "Logistic Regression", "14_lr_metrics_by_split.png")
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,
                          "Logistic Regression", f"Split {len(splits)}",
                          "15_lr_feature_coefficients.png")

print("\nLogistic Regression complete.")

## Phase 7 — Algorithm 2: Random Forest (placeholder)

_Implementation pending._ Drop the model code into the cell below — it should mirror the LR section:

1. Build a `RandomForestClassifier` (e.g. `n_estimators=200`, `class_weight=None` since SMOTE handles balance, `random_state=RANDOM_STATE`, `n_jobs=-1`).
2. Loop over `splits`, call `scale_and_resample`, fit, predict, `compute_metrics`, store row.
3. Persist `rf_df.to_csv("results/rf_results.csv", ...)` and call `print_summary_table`.
4. Visualisations: `plot_confusion_matrices`, `plot_roc_curves`, `plot_metrics_bars`, plus a feature-importance bar chart from `model.feature_importances_` (filenames `16`–`19`).

In [ ]:
# TODO: Random Forest training & evaluation
# from sklearn.ensemble import RandomForestClassifier
#
# rf_results = []
# rf_last_model = None
# for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
#     X_tr, y_tr, X_te, y_te, _ = scale_and_resample(X_features, y, train_idx, test_idx)
#     model = RandomForestClassifier(
#         n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1,
#     )
#     model.fit(X_tr, y_tr)
#     y_pred = model.predict(X_te)
#     y_prob = model.predict_proba(X_te)[:, 1]
#     metrics = compute_metrics(y_te, y_pred, y_prob)
#     metrics["split"] = split_num
#     rf_results.append(metrics)
#     rf_last_model = model
#
# rf_df = pd.DataFrame(rf_results)
# rf_df.to_csv(os.path.join(RESULTS_DIR, "rf_results.csv"), index=False)
# print_summary_table(rf_df, "Random Forest")
pass

In [ ]:
# TODO: Random Forest visualisations (16_rf_confusion_matrices.png, 17_rf_roc_curves.png,
#       18_rf_metrics_by_split.png, 19_rf_feature_importances.png)
pass

## Phase 8 — Algorithm 3: Feed-Forward Neural Network

Two hidden layers (64 → 32) with BatchNorm + Dropout, trained with Adam + BCE loss.  
Architecture: `Input(29) → Dense(64, ReLU, BN, Drop) → Dense(32, ReLU, BN, Drop) → Dense(1, Sigmoid)`.

In [ ]:
# Hyperparameters
HIDDEN1 = 64
HIDDEN2 = 32
DROPOUT = 0.3
EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


class FraudDetectorNN(nn.Module):
    """FFNN: Input -> 64 -> 32 -> 1, with BatchNorm + ReLU + Dropout."""

    def __init__(self, input_dim, hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


def _set_seeds():
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)


def _train_nn(X_train, y_train, input_dim, epochs=EPOCHS, batch_size=BATCH_SIZE,
              lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    """Train an FFNN on (X_train, y_train) and return (model, epoch_losses)."""
    _set_seeds()

    model = FraudDetectorNN(input_dim).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    X_arr = X_train.values if hasattr(X_train, "values") else np.array(X_train)
    y_arr = y_train.values if hasattr(y_train, "values") else np.array(y_train)
    X_tensor = torch.FloatTensor(X_arr)
    y_tensor = torch.FloatTensor(y_arr)

    dataset = TensorDataset(X_tensor, y_tensor)
    g = torch.Generator().manual_seed(RANDOM_STATE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=g)

    epoch_losses = []
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        n_samples = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            y_hat = model(X_batch)
            loss = criterion(y_hat, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(X_batch)
            n_samples += len(X_batch)

        avg_loss = running_loss / n_samples
        epoch_losses.append(avg_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"      Epoch {epoch + 1:>3}/{epochs}  |  Loss: {avg_loss:.6f}")

    return model, epoch_losses


def _predict(model, X_test):
    """Return (y_pred, y_prob) for the trained model."""
    model.eval()
    with torch.no_grad():
        X_arr = X_test.values if hasattr(X_test, "values") else np.array(X_test)
        X_tensor = torch.FloatTensor(X_arr).to(DEVICE)
        y_prob = model(X_tensor).cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob

In [ ]:
# ── FFNN training & evaluation across all splits ──
print("=" * 60)
print("9. FEED-FORWARD NEURAL NETWORK - TRAINING & EVALUATION")
print("=" * 60)

input_dim = X_features.shape[1]

print(f"\n  Architecture: Input({input_dim}) -> Dense({HIDDEN1}, ReLU, BN, Drop) "
      f"-> Dense({HIDDEN2}, ReLU, BN, Drop) -> Dense(1, Sigmoid)")
print(f"  Optimiser:    Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"  Loss:         Binary Cross-Entropy")
print(f"  Epochs: {EPOCHS}  |  Batch Size: {BATCH_SIZE}  |  Dropout: {DROPOUT}")
print(f"  Device:       {DEVICE}")

nn_results = []
split_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"\n  -- Split {split_num} --")

    X_train_res, y_train_res, X_test_scaled, y_test, _ = scale_and_resample(
        X_features, y, train_idx, test_idx,
    )
    print(f"    Train: {len(X_train_res):,} samples (after SMOTE)  |  "
          f"Test: {len(X_test_scaled):,} samples")

    model, epoch_losses = _train_nn(X_train_res, y_train_res, input_dim)
    y_pred, y_prob = _predict(model, X_test_scaled)

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)

    split_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

    print(f"    Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  "
          f"F1: {metrics['f1']:.4f}  |  MCC: {metrics['mcc']:.4f}  |  "
          f"AUC-ROC: {metrics['auc_roc']:.4f}")
    print_classification_report(y_test, y_pred)

nn_df = pd.DataFrame(nn_results)
nn_df.to_csv(os.path.join(RESULTS_DIR, "nn_results.csv"), index=False)
print_summary_table(nn_df, "Feed-Forward Neural Network")

In [ ]:
# ── FFNN visualisations ──
print("=" * 60)
print("10. FEED-FORWARD NEURAL NETWORK - VISUALISATIONS")
print("=" * 60)

# Training loss curves
fig, ax = plt.subplots(figsize=(10, 5))
colors_splits = ["#3498db", "#f39c12", "#2ecc71", "#e74c3c"]
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    color = colors_splits[(split_num - 1) % len(colors_splits)]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Split {split_num}", color=color, linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "20_nn_training_loss.png")


# Re-use stored predictions to avoid retraining the NN per plot.
def make_predictor(key):
    predictions = [split_predictions[i + 1][key] for i in range(len(splits))]
    call_idx = [0]

    def predictor(train_idx, test_idx):
        result = predictions[call_idx[0]]
        call_idx[0] += 1
        return result

    return predictor


plot_confusion_matrices(splits, X_features, y, make_predictor("y_pred"),
                        "Feed-Forward Neural Network", "21_nn_confusion_matrices.png")
plot_roc_curves(splits, X_features, y, make_predictor("y_prob"),
                "Feed-Forward Neural Network", "22_nn_roc_curves.png")
plot_metrics_bars(nn_df, splits, "Feed-Forward Neural Network",
                  "23_nn_metrics_by_split.png")

print("\nFeed-Forward Neural Network complete.")

## Phase 9 — Cross-model summary

In [ ]:
summary_frames = {
    "Logistic Regression": lr_df,
    "Feed-Forward NN": nn_df,
}
# When Random Forest is implemented, add: summary_frames["Random Forest"] = rf_df

metric_cols = ["precision", "recall", "f1", "mcc", "auc_roc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()
    for name, frame in summary_frames.items()
}).T

print("Mean metrics across all time-based splits:")
print(summary.round(4))

## Phase 10 — Download artefacts (optional)

All plots and CSVs are written to `./results/` in the Colab session. To pull them down locally:

```python
import shutil
from google.colab import files
shutil.make_archive("dataset3_results", "zip", "results")
files.download("dataset3_results.zip")
```